# Shared Housing: a multilingual Singapore household

This example simulates four residents of a Clementi HDB flat. The personas define habits, language repertoires, conflict styles, and asymmetric relationships. Episode triggers establish only the situation and communication channel; the residents' reactions are generated by TinyTroupe.

In [ ]:
%%capture
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

repo_root = Path.cwd().parent if (Path.cwd().parent / 'tinytroupe').is_dir() else Path.cwd()
sys.path.insert(0, str(repo_root))
load_dotenv(find_dotenv(usecwd=True))

import tinytroupe.control as control
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.examples import create_shared_housing_scenario, extract_shared_housing_signals

# These resets make this setup cell safe to rerun in the same notebook kernel.
control.reset()
TinyPerson.clear_agents()
TinyWorld.clear_environments()
TinyPerson.communication_display = False
TinyWorld.communication_display = False

scenario = create_shared_housing_scenario()
world = scenario.world

## Meet the household

In [ ]:
from IPython.display import Markdown, display

resident_profiles = ['## Household profiles']
for resident in scenario.residents.values():
    resident_profiles.extend([f'### {resident.name}', resident.minibio(), ''])
display(Markdown('\n\n'.join(resident_profiles)))

## Run a household timeline

All eleven situations from the design document are available. This run uses six episodes in chronological order and preserves each resident's memory between them, allowing later responses to reflect earlier interactions.

In [ ]:
episode_rows = [
    '| Key | Category | Episode |',
    '|---|---|---|',
]
for episode in scenario.available_episodes:
    episode_rows.append(f'| `{episode.key}` | {episode.category.title()} | {episode.title} |')
display(Markdown('\n'.join(episode_rows)))

In [ ]:
episode_plan = [
    'chinese_new_year',
    'late_kitchen_incident',
    'ravi_guest_afternoon',
    'maria_night_shift_return',
    'maintenance_request',
    'deepavali_decorations',
]

episode_openings = {
    'chinese_new_year': "Wei Ling's parents arrive for a three-day visit. Her mother speaks Mandarin and Hokkien but almost no English.",
    'late_kitchen_incident': 'Wei Ling finds fresh oil on the stove and posts: “Please clean the stove after cooking, thanks.”',
    'ravi_guest_afternoon': "Three of Ravi's colleagues have occupied the living room loudly for almost four hours.",
    'maria_night_shift_return': "After silently clearing Ravi's guests' containers at 12:30 AM, Maria tells Wei Ling privately the next morning.",
    'maintenance_request': 'The shared shower head breaks; someone must coordinate with the landlord and relay the plan.',
    'deepavali_decorations': 'Ravi puts up corridor lights and a paper lantern for Deepavali without consulting the household first.',
}

episode_runs = []
for episode_key in episode_plan:
    episode_actions = scenario.run_episode(
        episode_key,
        steps=2,
        parallelize=True,
        randomize_agents_order=False,
    )
    episode_runs.append((scenario.current_episode.definition, episode_actions))

def clean_markdown_text(value):
    return ' '.join(str(value or '').split())

transcript = ['## Household interaction timeline', '']
for definition, actions in episode_runs:
    transcript.extend([
        f'### {definition.title}',
        '',
        f'**Starting situation:** {episode_openings[definition.key]}',
        '',
    ])
    for step_number, step_actions in enumerate(actions, start=1):
        transcript.extend([f'#### Interaction step {step_number}', ''])
        for resident_name, records in step_actions.items():
            for record in records:
                action = record.get('action', {})
                action_type = action.get('type', 'ACTION')
                content = clean_markdown_text(action.get('content'))
                target = clean_markdown_text(action.get('target'))
                if action_type == 'DONE':
                    continue
                if action_type == 'THINK':
                    transcript.extend([
                        f'<details><summary><strong>{resident_name} — private thought</strong></summary>',
                        '',
                        content,
                        '',
                        '</details>',
                        '',
                    ])
                elif action_type == 'TALK':
                    destination = f' → {target}' if target else ''
                    transcript.append(f'- **{resident_name}{destination}:** {content}')
                else:
                    transcript.append(f'- **{resident_name} · {action_type}:** {content}')
        transcript.append('')
display(Markdown('\n'.join(transcript)))

This notebook intentionally reuses one scenario so memories and relationship trajectories carry across episodes. For isolated comparisons, create a fresh `create_shared_housing_scenario()` before each episode instead.

## Extract the research signals

The extractor distinguishes observed evidence from interpretation across language accommodation, grouping, conflict handling, translation, and linguistic social distance.

In [ ]:
signals = extract_shared_housing_signals(scenario)

signal_labels = {
    'language_accommodation': 'Language accommodation',
    'social_grouping': 'Social grouping',
    'conflict_resolution': 'Conflict resolution',
    'spontaneous_translation': 'Spontaneous translation',
    'language_as_social_distance': 'Language as social distance',
}
report = ['## Extracted research signals', '']
for field, label in signal_labels.items():
    report.extend([f'### {label}', ''])
    finding = signals.get(field)
    if finding is None:
        report.extend(['_No evidence observed in this episode._', ''])
    elif isinstance(finding, dict):
        observed = clean_markdown_text(finding.get('observed'))
        interpretation = clean_markdown_text(finding.get('interpretation'))
        report.extend([
            f'**Observed:** {observed}',
            '',
            f'**Interpretation:** {interpretation}',
            '',
        ])
    else:
        report.extend([clean_markdown_text(finding), ''])
display(Markdown('\n'.join(report)))